# ai.wasm を Python から使うサンプル

## 概要

`ai.wasm` は Emscripten でコンパイルされたオセロAI（Egaroucid）です。  
WASM 本体と JS グルーコード（`ai.js`）がセットで動作するため、  
Python から直接 WASM を読む代わりに **Node.js をサブプロセスとして呼び出す**方式を採用しています。

## 前提
- Node.js がインストールされていること
- `js/ai.wasm` と `js/ai.js` がこのノートブックの隣の `js/` フォルダにあること

## 盤面の表現（Python側）
```
board[y][x]  ← 8×8 の二次元リスト
 1  = 黒石
-1  = 白石
 0  = 空マス

座標: (x=0, y=0) が左上 (a1)、(x=7, y=7) が右下 (h8)
```

## WASM の主なエクスポート関数
| 関数 | 引数 | 返り値 |
|------|------|--------|
| `_init_ai(ptr)` | 空ポインタ | 0=成功 |
| `_ai_js(boardPtr, level, player)` | 盤面・深さ・手番 | エンコード済み整数 |
| `_calc_value(boardPtr, level, player)` | 同上 | 評価値のみ |

---
## 1. 準備：Node.js の確認

In [ ]:
import subprocess
import json
import os
import tempfile
from pathlib import Path

# このノートブックが置かれているディレクトリを基準にパスを設定
NOTEBOOK_DIR = Path(os.getcwd())
AI_JS_PATH   = NOTEBOOK_DIR / 'js' / 'ai.js'

# Node.js バージョン確認
result = subprocess.run(['node', '--version'], capture_output=True, text=True)
if result.returncode == 0:
    print(f'Node.js: {result.stdout.strip()}')
else:
    raise RuntimeError('Node.js が見つかりません。インストールしてください。')

# ai.js の存在確認
if AI_JS_PATH.exists():
    print(f'ai.js: {AI_JS_PATH}')
else:
    raise FileNotFoundError(f'ai.js が見つかりません: {AI_JS_PATH}')

---
## 2. Node.js ブリッジスクリプトの定義

Python から `subprocess` で呼ぶ Node.js スクリプトを一時ファイルに書き出します。  
このスクリプトは:
1. ai.js を `require()` して AI を初期化
2. コマンドライン引数で受け取った盤面を評価
3. 結果を JSON で stdout に出力

In [ ]:
NODE_BRIDGE_SCRIPT = r"""
'use strict';
const path = require('path');
const vm   = require('vm');
const fs   = require('fs');
const util = require('util');

// コマンドライン引数: node bridge.js <ai.js のパス> <JSON コマンド>
const [aiJsPath, cmdJson] = process.argv.slice(2);
const cmd = JSON.parse(cmdJson);

// ===== ヘルパー: 結果を JSON で stdout に出力して終了 =====
function output(obj) {
    process.stdout.write(JSON.stringify(obj) + '\n');
    process.exit(0);
}

// ===== 盤面エンコード =====
// Python 側の board[y][x] (1=黒,-1=白,0=空) を WASM 用 Int32Array に変換して
// HEAP に書き込み、ポインタを返す。使用後は M._free() すること。
function encodeBoard(M, board) {
    const arr = new Int32Array(64);
    for (let y = 0; y < 8; y++) {
        for (let x = 0; x < 8; x++) {
            const v = board[y][x];
            arr[y * 8 + x] = v === 1 ? 0 : v === -1 ? 1 : -1;
        }
    }
    const ptr = M._malloc(64 * 4);
    M.HEAP32.set(arr, ptr >> 2);
    return ptr;
}

// ===== _ai_js の戻り値をデコード =====
// 戻り値 val = my*8000 + mx*1000 + (score_raw + 100)
function decodeAiResult(val) {
    const my        = Math.floor(val / 8000);
    const mx        = Math.floor((val - my * 8000) / 1000);
    const score_raw = val - my * 8000 - mx * 1000 - 100;
    return { mx, my, score_raw };
}

// ===== コマンドディスパッチ =====
function dispatch(M, cmd) {
    if (cmd.type === 'best_move') {
        // 最善手と評価値を返す
        const wasmPlayer = cmd.player === 1 ? 0 : 1;
        const ptr = encodeBoard(M, cmd.board);
        const val = M._ai_js(ptr, cmd.level || 8, wasmPlayer);
        M._free(ptr);
        const { mx, my, score_raw } = decodeAiResult(val);
        const score = wasmPlayer === 0 ? score_raw : -score_raw;
        return { mx, my, score, col: 'abcdefgh'[mx], row: my + 1 };

    } else if (cmd.type === 'calc_value') {
        // 評価値のみを返す。
        // _calc_value は WASM にエクスポートされていない場合があるため、
        // _ai_js を呼んで座標を無視しスコアだけ取り出す。
        const wasmPlayer = cmd.player === 1 ? 0 : 1;
        const ptr = encodeBoard(M, cmd.board);
        const val = M._ai_js(ptr, cmd.level || 8, wasmPlayer);
        M._free(ptr);
        const { score_raw } = decodeAiResult(val);
        const score = wasmPlayer === 0 ? score_raw : -score_raw;
        return { score };

    } else {
        return { error: 'unknown command type: ' + cmd.type };
    }
}

// ===== AI 初期化 + コマンド実行 =====
// Emscripten は `Module.onRuntimeInitialized()` と呼ぶので this === Module
function onReady() {
    const M = this;
    try {
        const initPtr = M._malloc(4);
        const initResult = M._init_ai(initPtr);
        M._free(initPtr);
        if (initResult !== 0) {
            output({ error: '_init_ai failed: ' + initResult });
            return;
        }
        output(dispatch(M, cmd));
    } catch(e) {
        output({ error: String(e) });
    }
}

// ===== vm サンドボックスで ai.js を実行 =====
//
// 問題の背景:
//   ai.js の先頭に `var Module = typeof Module != "undefined" ? Module : {};` がある。
//   Node.js の require() はファイルを関数ラッパーで囲むため、`var Module` は
//   ラッパー関数ローカルの変数になる。hoisting により typeof Module は
//   常に "undefined" → {} が作られてしまい、事前に global.Module を仕込んでも届かない。
//
// 解決策:
//   vm.runInContext() でファイルをグローバルスコープとして実行する。
//   `var Module` が sandbox のグローバル Module プロパティを参照するため、
//   事前に定義した onRuntimeInitialized コールバックが確実に使われる。

const aiJsSource = fs.readFileSync(path.resolve(aiJsPath), 'utf8');
const aiDir      = path.dirname(path.resolve(aiJsPath));

const sandbox = {
    process,
    Buffer,
    setTimeout, clearTimeout,
    setInterval, clearInterval,
    setImmediate, clearImmediate,
    console: { log: ()=>{}, warn: ()=>{}, error: ()=>{}, info: ()=>{} },
    require: (id) => {
        if (id.startsWith('.')) return require(path.resolve(aiDir, id));
        return require(id);
    },
    __filename: path.resolve(aiJsPath),
    __dirname:  aiDir,
    WebAssembly,
    TextDecoder: typeof TextDecoder !== 'undefined' ? TextDecoder : util.TextDecoder,
    TextEncoder: typeof TextEncoder !== 'undefined' ? TextEncoder : util.TextEncoder,
    performance: typeof performance !== 'undefined'
        ? performance
        : require('perf_hooks').performance,
    Module: {
        print:                () => {},
        printErr:             () => {},
        onRuntimeInitialized: onReady,
    },
};

vm.createContext(sandbox);
vm.runInContext(aiJsSource, sandbox);
"""

import tempfile
from pathlib import Path

BRIDGE_FILE = Path(tempfile.mktemp(suffix='.js'))
BRIDGE_FILE.write_text(NODE_BRIDGE_SCRIPT, encoding='utf-8')
print(f'ブリッジスクリプト: {BRIDGE_FILE}')

---
## 3. Python ヘルパー関数

In [ ]:
def call_ai(cmd: dict, timeout: int = 30) -> dict:
    """Node.js ブリッジを呼び出し、結果を dict で返す。"""
    result = subprocess.run(
        ['node', str(BRIDGE_FILE), str(AI_JS_PATH), json.dumps(cmd)],
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    # stdout・stderr 両方をデバッグ用に確認できるようにする
    if result.returncode != 0 or not result.stdout.strip():
        raise RuntimeError(
            f'Node.js 失敗\n'
            f'  returncode : {result.returncode}\n'
            f'  stdout     : {repr(result.stdout[:300])}\n'
            f'  stderr     : {repr(result.stderr[:500])}'
        )
    return json.loads(result.stdout)


def best_move(board: list[list[int]], player: int, level: int = 8) -> dict:
    """
    Egaroucid に最善手を問い合わせる。

    Parameters
    ----------
    board  : 8×8 の二次元リスト (1=黒, -1=白, 0=空)
    player : 手番 (1=黒, -1=白)
    level  : 探索深さ (1〜21、デフォルト8)

    Returns
    -------
    dict: { mx: int, my: int, score: int, col: str, row: int }
          mx/my は 0-indexed 座標、col/row は棋譜表記 (例: 'd', 3)
    """
    return call_ai({'type': 'best_move', 'board': board, 'player': player, 'level': level})


def calc_value(board: list[list[int]], player: int, level: int = 8) -> int:
    """
    Egaroucid に局面の評価値（黒視点の予測石差）を問い合わせる。

    Returns
    -------
    int: 正=黒有利, 負=白有利
    """
    result = call_ai({'type': 'calc_value', 'board': board, 'player': player, 'level': level})
    return result['score']


# ===== 盤面ユーティリティ =====

def initial_board() -> list[list[int]]:
    """オセロの初期盤面を返す（中央4マスに初期配置）。"""
    b = [[0] * 8 for _ in range(8)]
    b[3][3] = -1  # d4: 白
    b[3][4] =  1  # e4: 黒
    b[4][3] =  1  # d5: 黒
    b[4][4] = -1  # e5: 白
    return b


def print_board(board: list[list[int]], last_move: tuple[int,int] | None = None) -> None:
    """盤面をテキストで表示する。"""
    print('  a b c d e f g h')
    for y, row in enumerate(board):
        line = f'{y+1} '
        for x, v in enumerate(row):
            if last_move and (x, y) == last_move:
                line += ('● ' if v == 1 else '○ ')
            elif v == 1:
                line += '● '
            elif v == -1:
                line += '○ '
            else:
                line += '. '
        print(line)
    black = sum(v == 1  for row in board for v in row)
    white = sum(v == -1 for row in board for v in row)
    print(f'   黒: {black}  白: {white}')


def get_flips(board, x, y, player):
    """(x,y) に player が打ったときにひっくり返る座標のリストを返す。"""
    opponent = -player
    flips = []
    for dx, dy in [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]:
        line = []
        nx, ny = x + dx, y + dy
        while 0 <= nx < 8 and 0 <= ny < 8 and board[ny][nx] == opponent:
            line.append((nx, ny))
            nx += dx; ny += dy
        if line and 0 <= nx < 8 and 0 <= ny < 8 and board[ny][nx] == player:
            flips.extend(line)
    return flips


def apply_move(board, x, y, player):
    """(x,y) に player が打った後の新しい盤面を返す（元の盤面は変更しない）。"""
    import copy
    b = copy.deepcopy(board)
    b[y][x] = player
    for fx, fy in get_flips(board, x, y, player):
        b[fy][fx] = player
    return b


def valid_moves(board, player):
    """player が打てるすべての (x, y) を返す。"""
    return [(x, y)
            for y in range(8)
            for x in range(8)
            if board[y][x] == 0 and get_flips(board, x, y, player)]


print('ヘルパー関数定義完了')

---
## 4. 動作確認：初期局面の最善手を求める

In [ ]:
board = initial_board()
print('=== 初期局面 ===')
print_board(board)

print('\n黒番の最善手を計算中 (level=8)...')
result = best_move(board, player=1, level=8)
print(f'最善手: {result["col"]}{result["row"]}  (x={result["mx"]}, y={result["my"]})')
print(f'評価値: {result["score"]:+d}  (黒視点の予測石差)')

---
## 5. 評価値のみを取得する

In [ ]:
board = initial_board()

# 黒の合法手それぞれを打った後の局面を評価
moves = valid_moves(board, player=1)
print('黒の合法手と評価値:')
print(f'{"手":>5}  {"評価値":>6}')
print('-' * 14)

for x, y in moves:
    # 一手打った後の局面で白番として評価（次は白の手番なので）
    next_board = apply_move(board, x, y, player=1)
    score = calc_value(next_board, player=-1, level=8)
    col = 'abcdefgh'[x]
    print(f'{col}{y+1:>4}  {score:>+6d}')

---
## 6. AI 同士を対局させる

---
## 6. 終盤全読み（15手）

空きマスが少ない終盤局面では、Egaroucid に高いレベルを渡すことで完全読み（全読み）が可能です。  
両者最善手を交互に打ち切り、最終的な実石差を返します。

| レベル目安 | 空きマス数 |
|-----------|-----------|
| 10 | ≦20 |
| 15 | ≦17 |
| 21（全読み） | ≦15 |

In [ ]:
def solve_endgame(board: list[list[int]], player: int, level: int = 21) -> dict:
    """
    終盤局面を全読みして最終結果を返す。

    両者最善手を交互に打ち切り、最後の実石差（日本ルール: 勝者に空きマスを加算）を返す。

    Parameters
    ----------
    board  : 8×8 の二次元リスト (1=黒, -1=白, 0=空)
    player : 手番 (1=黒, -1=白)
    level  : 探索レベル（21 = 全読み推奨）

    Returns
    -------
    dict:
        score      : 最終石差（黒視点: 正=黒勝, 負=白勝, 0=引き分け）
        black      : 最終黒石数（日本ルール補正後）
        white      : 最終白石数（日本ルール補正後）
        moves      : 最善手順リスト [{'col':str, 'row':int, 'player':int}, ...]
        final_board: 打ち切り後の最終盤面
    """
    import copy
    b      = copy.deepcopy(board)
    cp     = player
    moves  = []
    passes = 0

    while True:
        vm = valid_moves(b, cp)
        if not vm:
            # 合法手なし → パス or 終局
            if not valid_moves(b, -cp):
                break  # 両者パス → 終局
            passes += 1
            if passes > 1:
                break
            cp = -cp
            continue
        passes = 0

        # Egaroucid に最善手を問い合わせ
        r  = best_move(b, player=cp, level=level)
        x, y = r['mx'], r['my']
        moves.append({'col': r['col'], 'row': r['row'], 'player': cp})
        b  = apply_move(b, x, y, cp)
        cp = -cp

    # 最終石数を集計（日本ルール: 空きマスは勝者に加算）
    black = sum(v == 1  for row in b for v in row)
    white = sum(v == -1 for row in b for v in row)
    empty = sum(v == 0  for row in b for v in row)
    if black > white:
        black += empty
    elif white > black:
        white += empty
    # 引き分けは空きを半分ずつ（整数なら均等に）

    return {
        'score':       black - white,
        'black':       black,
        'white':       white,
        'moves':       moves,
        'final_board': b,
    }


# ===== サンプル終盤局面（空きマス 15 個）=====
#
#   a b c d e f g h
# 1 ● ● ● ● ● ● ● ●
# 2 ● ○ ○ ○ ○ ○ ○ ●
# 3 ● ○ ● ● ● ● ○ ●
# 4 ● ○ ● ○ ○ ● ○ ●
# 5 ● ○ ● ● ○ ● ○ .   ← h5 が空き
# 6 ● ○ ○ ● ○ . . .   ← f6 g6 h6 が空き
# 7 ● ● ● ● . . . .   ← e7 f7 g7 h7 が空き
# 8 ● . . . . . . .   ← b8〜h8 が空き
#
# 合計: 1+3+4+7 = 15 空き  黒31手・白18手

endgame_board = [
    [ 1,  1,  1,  1,  1,  1,  1,  1],  # row 1
    [ 1, -1, -1, -1, -1, -1, -1,  1],  # row 2
    [ 1, -1,  1,  1,  1,  1, -1,  1],  # row 3
    [ 1, -1,  1, -1, -1,  1, -1,  1],  # row 4
    [ 1, -1,  1,  1, -1,  1, -1,  0],  # row 5: h5=空き
    [ 1, -1, -1,  1, -1,  0,  0,  0],  # row 6: f6,g6,h6=空き
    [ 1,  1,  1,  1,  0,  0,  0,  0],  # row 7: e7,f7,g7,h7=空き
    [ 1,  0,  0,  0,  0,  0,  0,  0],  # row 8: b8〜h8=空き
]
empty_count = sum(v == 0 for row in endgame_board for v in row)

print('=== 終盤局面（黒番）===')
print_board(endgame_board)
print(f'空きマス数: {empty_count}')

print(f'\nレベル21 で全読み中...')
result = solve_endgame(endgame_board, player=1, level=21)

print('\n=== 最善手順 ===')
for i, m in enumerate(result['moves']):
    side = '黒' if m['player'] == 1 else '白'
    print(f'  手{i+1:>2}: {side} → {m["col"]}{m["row"]}')

print('\n=== 最終盤面 ===')
print_board(result['final_board'])

score = result['score']
winner = '黒' if score > 0 else '白' if score < 0 else '引き分け'
print(f'\n最終結果: 黒 {result["black"]} - 白 {result["white"]}')
print(f'勝者: {winner}  (石差: {score:+d})')

In [ ]:
def play_game(level: int = 5, max_moves: int = 60) -> list[list[int]]:
    """
    Egaroucid AI 同士でオセロを対局させ、最終盤面を返す。

    Parameters
    ----------
    level     : 探索深さ（小さいほど速い）
    max_moves : 最大手数
    """
    board  = initial_board()
    player = 1  # 黒先手
    history = []

    for move_num in range(max_moves):
        moves = valid_moves(board, player)

        if not moves:
            # 合法手なし → 相手にパス
            opp_moves = valid_moves(board, -player)
            if not opp_moves:
                print(f'終局（{move_num}手目）')
                break
            side = '黒' if player == 1 else '白'
            print(f'{side} パス')
            player = -player
            continue

        # AI に最善手を問い合わせ
        result = best_move(board, player=player, level=level)
        x, y = result['mx'], result['my']
        col   = 'abcdefgh'[x]
        side  = '黒' if player == 1 else '白'
        print(f'手{move_num+1:>2}: {side} → {col}{y+1}  (評価値: {result["score"]:+d})')

        board  = apply_move(board, x, y, player)
        history.append((x, y, player))
        player = -player

    return board


print('=== AI対局開始 (level=5) ===')
final_board = play_game(level=5)
print('\n=== 最終盤面 ===')
print_board(final_board)

---
## 7. 任意の局面を指定して評価する

棋譜を手動で入力し、その局面を評価する例です。

In [ ]:
def board_from_kifu(kifu: str) -> tuple[list[list[int]], int]:
    """
    棋譜文字列から盤面と手番を復元する。

    Parameters
    ----------
    kifu : 棋譜文字列。例: 'd3c3c4d6c5f6e6f5f4f3e3e7c6g5'
           小文字 'a'〜'h' + 数字 '1'〜'8' を交互に並べる。

    Returns
    -------
    (board, player) : 最終局面と次の手番
    """
    board  = initial_board()
    player = 1  # 黒先手

    i = 0
    while i + 1 < len(kifu):
        col_ch = kifu[i]
        row_ch = kifu[i + 1]
        i += 2

        x = 'abcdefgh'.index(col_ch)
        y = int(row_ch) - 1

        flips = get_flips(board, x, y, player)
        if not flips:
            # 合法手でない場合はパスとみなして手を再試行
            player = -player
            flips  = get_flips(board, x, y, player)

        board  = apply_move(board, x, y, player)
        player = -player

    return board, player


# --- 実行例 ---
# 有名な「花月定石」の一部を入力
kifu = 'd3c4d2c3e2f2'
board, next_player = board_from_kifu(kifu)

print(f'棋譜: {kifu}')
print(f'次の手番: {"黒" if next_player == 1 else "白"}')
print_board(board)

# この局面の最善手と評価値を問い合わせ
result = best_move(board, player=next_player, level=10)
print(f'\nEgaroucid の最善手: {result["col"]}{result["row"]}  評価値: {result["score"]:+d}')

---
## 8. クリーンアップ

In [ ]:
# 一時ブリッジファイルを削除
if BRIDGE_FILE.exists():
    BRIDGE_FILE.unlink()
    print(f'削除: {BRIDGE_FILE}')